# Cinema Revenue Prediction with Apache Spark

このNotebookでは、Apache Sparkを使用してCinemaデータセットを探索し、興行収入予測の回帰モデルを構築します。

In [ ]:
// Spark 依存関係の読み込み（Scala 2.13 を明示的に指定）
import $ivy.`org.apache.spark:spark-sql_2.13:3.5.0`
import $ivy.`org.apache.spark:spark-mllib_2.13:3.5.0`

println("Spark 依存関係が正常にロードされました")

## 1. 環境設定とライブラリのインポート

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.ml.feature.{StringIndexer, OneHotEncoder, VectorAssembler}
import org.apache.spark.ml.evaluation.RegressionEvaluator

// SparkSessionの作成
val spark = SparkSession.builder()
  .appName("CinemaExploration")
  .master("local[*]")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

println("Spark Session created successfully!")
println(s"Spark version: ${spark.version}")

## 2. データの読み込み

In [ ]:
// データの読み込み
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("../data/cinema.csv")

println(s"データ件数: ${df.count()}")
println("\nスキーマ:")
df.printSchema()

## 3. データの概要確認

In [ ]:
// 最初の5行を表示
df.show(5, truncate = false)

In [ ]:
// 統計情報
df.describe("budget", "popularity", "runtime", "vote_average", "revenue").show()

In [ ]:
// ジャンルごとの件数
df.groupBy("genre").count().orderBy($"count".desc).show()

## 4. ジャンルのOneHotエンコーディング

In [ ]:
// ステップ1: StringIndexer でジャンルを数値に変換
val indexer = new StringIndexer()
  .setInputCol("genre")
  .setOutputCol("genre_index")

// ステップ2: OneHotEncoder でダミー変数化
val encoder = new OneHotEncoder()
  .setInputCol("genre_index")
  .setOutputCol("genre_vec")
  .setDropLast(false)

val encodePipeline = new Pipeline().setStages(Array(indexer, encoder))
val encodedDf = encodePipeline.fit(df).transform(df)

println("ジャンルのエンコーディング完了")
encodedDf.select("genre", "genre_index", "genre_vec").show(5, truncate = false)

## 5. 特徴量の統合

In [ ]:
// 全ての特徴量を1つのベクトルに統合
val assembler = new VectorAssembler()
  .setInputCols(Array(
    "budget",
    "popularity",
    "runtime",
    "vote_average",
    "genre_vec"
  ))
  .setOutputCol("features")
  .setHandleInvalid("skip")

val assembledDf = assembler.transform(encodedDf)

println(s"準備後のデータ件数: ${assembledDf.count()}")
assembledDf.select("features", "revenue").show(5, truncate = false)

## 6. データの分割

In [ ]:
// 訓練データとテストデータに分割
val Array(trainData, testData) = assembledDf.randomSplit(Array(0.7, 0.3), seed = 42)

println(s"訓練データ: ${trainData.count()} 件")
println(s"テストデータ: ${testData.count()} 件")

## 7. Linear Regressionモデルの訓練

In [ ]:
// Linear Regressionモデルの作成
val lr = new LinearRegression()
  .setLabelCol("revenue")
  .setFeaturesCol("features")
  .setMaxIter(100)
  .setRegParam(0.1)
  .setElasticNetParam(0.0)

val pipeline = new Pipeline().setStages(Array(lr))

// モデルの訓練
println("モデルを訓練中...")
val model = pipeline.fit(trainData)
println("訓練完了！")

## 8. モデルの評価

In [ ]:
// テストデータで予測
val predictions = model.transform(testData)

// 評価メトリクスの計算
val evaluator = new RegressionEvaluator()
  .setLabelCol("revenue")
  .setPredictionCol("prediction")

val r2 = evaluator.setMetricName("r2").evaluate(predictions)
val rmse = evaluator.setMetricName("rmse").evaluate(predictions)
val mae = evaluator.setMetricName("mae").evaluate(predictions)

println(f"R² Score: ${r2 * 100}%.2f%%")
println(f"RMSE: $rmse%.2f")
println(f"MAE: $mae%.2f")

## 9. 予測結果の確認

In [ ]:
// 予測結果のサンプル表示
predictions.select(
  "budget", "popularity", "runtime", "vote_average", "genre",
  "revenue", "prediction"
).show(10, truncate = false)

In [ ]:
// 実際の値と予測値の比較（誤差を計算）
import org.apache.spark.sql.functions._

val comparison = predictions.select(
  col("revenue").as("actual"),
  col("prediction"),
  abs(col("revenue") - col("prediction")).as("error"),
  (abs(col("revenue") - col("prediction")) / col("revenue") * 100).as("error_pct")
)

comparison.describe("actual", "prediction", "error", "error_pct").show()

## 10. モデルの係数確認

In [ ]:
// Linear Regressionモデルの係数を表示
val lrModel = model.stages(0).asInstanceOf[org.apache.spark.ml.regression.LinearRegressionModel]

println("モデルの係数:")
println(s"Intercept: ${lrModel.intercept}")
println(s"Coefficients: ${lrModel.coefficients}")
println()
println("訓練セットでの性能:")
println(s"RMSE: ${lrModel.summary.rootMeanSquaredError}")
println(s"R²: ${lrModel.summary.r2}")

## 11. 特徴量の重要度確認

In [ ]:
// 係数の絶対値を特徴量の重要度として表示
val featureNames = Array("budget", "popularity", "runtime", "vote_average") ++ 
                   (0 until lrModel.coefficients.size - 4).map(i => s"genre_$i")

val coefficients = lrModel.coefficients.toArray

println("特徴量の重要度（係数の絶対値）:")
featureNames.zip(coefficients).sortBy(-_._2.abs).take(10).foreach { case (name, coef) =>
  println(f"$name%-20s: $coef%.4f")
}

## 12. クリーンアップ

In [ ]:
// SparkSessionの停止
// spark.stop()
println("完了！")